# Checkpoint 1 
## (Do not remove any comments that start with"# @@@".) 

Reminder: 

- You are being evaluated for completion and effort in this checkpoint. 
- Avoid manual labor / hard coding as much as possible, everything we've taught you so far are meant to simplify and automate your process.
- Please do not remove any comment that starts with: "# @@@". 

We will be working with the same `states_edu.csv` that you should already be familiar with from the tutorial.

We investigated Grade 8 reading score in the tutorial. For this checkpoint, you are asked to investigate another test. Here's an overview:

* Choose a specific response variable to focus on
>Grade 4 Math, Grade 4 Reading, Grade 8 Math
* Pick or create features to use
>Will all the features be useful in predicting test score? Are some more important than others? Should you standardize, bin, or scale the data?
* Explore the data as it relates to that test
>Create at least 2 visualizations (graphs), each with a caption describing the graph and what it tells us about the data
* Create training and testing data
>Do you want to train on all the data? Only data from the last 10 years? Only Michigan data?
* Train a ML model to predict outcome 
>Define what you want to predict, and pick a model in sklearn to use (see sklearn <a href="https://scikit-learn.org/stable/modules/linear_model.html">regressors</a>).


Include comments throughout your code! Every cleanup and preprocessing task should be documented.


<h2> Data Cleanup </h2>

Import `numpy`, `pandas`, and `matplotlib`.

(Feel free to import other libraries!)

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#doesnt hide error messages
pd.options.mode.chained_assignment = None

Load in the "states_edu.csv" dataset and take a look at the head of the data

In [ ]:
#load csv file
df = pd.read_csv("../data/states_edu.csv")

# check rows and columns
print("DataFrame shape (rows, columns):", df.shape)
df.head()

You should always familiarize yourself with what each column in the dataframe represents. Read about the states_edu dataset here: https://www.kaggle.com/noriuk/us-education-datasets-unification-project

Use this space to rename columns, deal with missing data, etc. _(optional)_

In [ ]:
# rename enrollment columns
df.rename({
    'GRADES_PK_G': 'ENROLL_PREK',
    'GRADES_KG_G': 'ENROLL_KINDER',
    'GRADES_4_G': 'ENROLL_4',
    'GRADES_8_G': 'ENROLL_8',
    'GRADES_12_G': 'ENROLL_12',
    'GRADES_1_8_G': 'ENROLL_PRIMARY',
    'GRADES_9_12_G': 'ENROLL_HS',
    'GRADES_ALL_G': 'ENROLL_ALL',
    'ENROLL': 'ENROLL_ALL_EST'
}, axis=1, inplace=True)

# fill missing enrollment
# keep all-missing rows as missing
enrollment_sum = df[['ENROLL_PREK', 'ENROLL_KINDER', 'ENROLL_PRIMARY', 'ENROLL_HS']].sum(axis=1, min_count=1)
df['ENROLL_ALL'] = df['ENROLL_ALL'].fillna(enrollment_sum)

# use the estimate as a backup
df['ENROLL_ALL_EST'] = df['ENROLL_ALL_EST'].fillna(df['ENROLL_ALL'])

# choose the response column
response_column = 'AVG_MATH_4_SCORE'

# keep rows for the year count
# drop needed missing values before modeling
print("Missing Grade 4 Math scores:", df[response_column].isna().sum())

<h2>Exploratory Data Analysis (EDA) </h2>

Chosen response variable: **Grade 4 Math** (`AVG_MATH_4_SCORE`).

I picked Grade 4 Math since Tutorial 1 used Grade 8 Reading. I used Grade 4 Reading and Grade 8 Math as related columns.

How many years of data are logged in our dataset? 

In [ ]:
# @@@ 1
# check: count unique years
number_of_years = df['YEAR'].nunique()
print("Number of distinct years in the dataset:", number_of_years)

Let's compare Michigan to Ohio. Which state has the higher average across all years in the test you chose?

In [ ]:
# @@@ 2
# compare state averages
michigan_ohio = df[df['STATE'].isin(['MICHIGAN', 'OHIO'])]
state_averages = michigan_ohio.groupby('STATE')[response_column].mean()
print("Average Grade 4 Math score by state:")
print(state_averages)
print("State with the higher average:", state_averages.idxmax())

Find the average for your chosen test across all states in 2019

In [ ]:
# @@@ 3
# get the 2019 average
average_2019 = df.loc[df['YEAR'] == 2019, response_column].mean()
print("Average Grade 4 Math score across states in 2019:", average_2019)

For each state, find a maximum value for your chosen test score

In [ ]:
# @@@ 4
# find each state's max
maximum_by_state = df.groupby('STATE')[response_column].max().sort_values(ascending=False)
print("Maximum Grade 4 Math score for each state:")
maximum_by_state

*Refer to the `Grouping and Aggregating` section in Tutorial 0 if you are stuck.

<h2> Feature Engineering </h2>

After exploring the data, you can choose to modify features that you would use to predict the performance of the students on your chosen response variable. 

You can also create your own features. For example, perhaps you figured that maybe a state's expenditure per student may affect their overall academic performance so you create a expenditure_per_student feature.

Use this space to modify or create features.

In [ ]:
# @@@ 5
# make spending per student
# compare states fairly
# avoid dividing by zero
df['INSTRUCTION_EXPENDITURE_PER_STUDENT'] = (
    df['INSTRUCTION_EXPENDITURE'].div(df['ENROLL_ALL'].replace(0, np.nan))
)

# check the new column
df[['INSTRUCTION_EXPENDITURE', 'ENROLL_ALL', 'INSTRUCTION_EXPENDITURE_PER_STUDENT']].head()

Feature engineering justification: I made `INSTRUCTION_EXPENDITURE_PER_STUDENT` by dividing instruction spending by enrollment. This helps compare bigger and smaller states. I left missing values alone and drop rows later if the model needs them. I left the values unscaled so the graphs are easier to read.

<h2>Visualization</h2>

Investigate the relationship between your chosen response variable and at least two predictors using visualizations. Write down your observations.

**Visualization 1**

In [ ]:
# @@@ 6
# compare reading and math
# drop missing points
viz1_data = df[['AVG_READING_4_SCORE', response_column]].dropna()

plt.figure(figsize=(8, 5))
plt.scatter(viz1_data['AVG_READING_4_SCORE'], viz1_data[response_column], alpha=0.7)
plt.xlabel('Grade 4 Reading Score')
plt.ylabel('Grade 4 Math Score')
plt.title('Grade 4 Reading and Grade 4 Math')
plt.tight_layout()
plt.show()

**Visualization 1 observation:** The points mostly go up. States with higher Grade 4 Reading scores usually have higher Grade 4 Math scores too.

**Visualization 2**

In [ ]:
# @@@ 7
# compare spending and math
# drop missing points
viz2_data = df[['INSTRUCTION_EXPENDITURE_PER_STUDENT', response_column]].dropna()

plt.figure(figsize=(8, 5))
plt.scatter(viz2_data['INSTRUCTION_EXPENDITURE_PER_STUDENT'], viz2_data[response_column], alpha=0.7)
plt.xlabel('Instruction Expenditure per Student')
plt.ylabel('Grade 4 Math Score')
plt.title('Per-Student Instruction Expenditure and Grade 4 Math')
plt.tight_layout()
plt.show()

**Visualization 2 observation:** This graph is more spread out. Spending per student seems to go up with math scores, but it is not the only thing that matters.

<h2> Data Creation </h2>

_Use this space to create train/test data_

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# @@@ 8
# X is inputs, y is the target
# check: keep the target out of X
predictor_columns = [
    'AVG_READING_4_SCORE',
    'AVG_MATH_8_SCORE',
    'INSTRUCTION_EXPENDITURE_PER_STUDENT'
]

# drop rows missing model values
model_data = df[predictor_columns + [response_column]].dropna().copy()
X = model_data[predictor_columns]
y = model_data[response_column]

print("Rows available for modeling:", len(model_data))
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

In [ ]:
# @@@ 9
# split into train and test
# test on unseen rows
# keep the split repeatable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

<h2> Prediction </h2>

ML Models [Resource](https://medium.com/@vijaya.beeravalli/comparison-of-machine-learning-classification-models-for-credit-card-default-data-c3cf805c9a5a)

In [ ]:
# @@@ 10
# import a regression model
from sklearn.linear_model import LinearRegression

In [ ]:
# @@@ 11
# create the model
model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

## Evaluation

Choose some metrics to evaluate the performance of your model, some of them are mentioned in the tutorial.

In [ ]:
# @@@ 12
# calculate model scores
# check prediction bias
# check average error
# check larger errors
r_squared = model.score(X_test, y_test)
mean_error = np.mean(y_pred - y_test)
mean_absolute_error = np.mean(np.abs(y_pred - y_test))
root_mean_squared_error = np.mean((y_pred - y_test) ** 2) ** 0.5

print("R-squared:", r_squared)
print("Mean error:", mean_error)
print("Mean absolute error:", mean_absolute_error)
print("Root mean squared error:", root_mean_squared_error)

We have copied over the graphs that visualize the model's performance on the training and testing set. 

Change `col_name` and modify the call to `plt.ylabel()` to isolate how a single predictor affects the model.

In [ ]:
# @@@ 13
# plot train predictions
col_name = 'AVG_READING_4_SCORE'

plt.figure(figsize=(12, 6))
plt.scatter(X_train[col_name], y_train, color='red')
plt.scatter(X_train[col_name], model.predict(X_train), color='green')
plt.legend(['True Training', 'Predicted Training'])
plt.xlabel(col_name)
plt.ylabel('Grade 4 Math Score')
plt.title('Model Behavior on Training Set')
plt.tight_layout()
plt.show()

In [ ]:
# @@@ 14
# plot test predictions
# check how it generalizes
col_name = 'AVG_READING_4_SCORE'

plt.figure(figsize=(12, 6))
plt.scatter(X_test[col_name], y_test, color='blue')
plt.scatter(X_test[col_name], model.predict(X_test), color='black')
plt.legend(['True Testing', 'Predicted Testing'])
plt.xlabel(col_name)
plt.ylabel('Grade 4 Math Score')
plt.title('Model Behavior on Testing Set')
plt.tight_layout()
plt.show()